# Loan Default Prediction

## 1. Project Overview

### Objective
Build a machine learning workflow to predict whether a borrower will default on a loan.

### Target Variable
- `default`
  - `1` = default
  - `0` = no default

### Notebook Goal
This notebook is a guided template. You will implement the core logic step by step using TODO prompts and hints.

## 2. Imports and Setup

Import core libraries for data handling, visualization, statistics, and modeling.

In [4]:
# TODO: Import common libraries for data science work.
# HINT: pandas, numpy, matplotlib.pyplot, seaborn
# HINT: sklearn modules (model_selection, preprocessing, metrics, linear_model, ensemble)
# HINT: scipy.stats for hypothesis testing

# Example starter imports (expand as needed):
import pandas as pd
import numpy as np

# TODO: Add plotting and ML imports here

# TODO: Set a reproducible random seed for your experiments
random_seed = 42
np.random.seed(random_seed)

## 3. Data Loading

Load your dataset and perform a quick structural inspection before any transformations.

# Loan Default Prediction: Data Dictionary

## Purpose
This dictionary standardizes field meaning for EDA, feature engineering, and modeling. Columns are grouped by analytical role rather than raw file order.

## 1) Entity Keys and Location
| Column | Description |
|---|---|
| id | Unique Lending Club identifier for the loan listing. |
| memberId | Unique Lending Club identifier for the borrower member. |
| url | Lending Club listing URL. |
| addrState | Borrower state from the application. |
| zip_code | First 3 digits of borrower ZIP code. |
| msa | Borrower Metropolitan Statistical Area. |

## 2) Application Lifecycle and Listing Status
| Column | Description |
|---|---|
| listD | Date the application was listed on the platform. |
| acceptD | Date borrower accepted the offer. |
| creditPullD | Date Lending Club pulled credit. |
| reviewStatus | Listing-period review status (APPROVED, NOT_APPROVED). |
| reviewStatusD | Date the application was reviewed. |
| expD | Listing expiration date. |
| ils_exp_d | Whole loan platform expiration date. |
| application_type | Individual vs joint application type. |
| initialListStatus | Initial listing status (W, F). |

## 3) Borrower Profile, Income, and Verification
| Column | Description |
|---|---|
| annualInc | Self-reported annual income (borrower). |
| annual_inc_joint | Combined self-reported annual income (co-borrowers). |
| isIncV | Income verification status for borrower. |
| verified_status_joint | Income verification status for joint applicants. |
| emp_title | Borrower job title at application. |
| empLength | Employment length in years (0 to 10, where 10 means 10+). |
| homeOwnership | Home ownership status (RENT, OWN, MORTGAGE, OTHER). |
| purpose | Borrower-provided loan purpose category. |
| title | Borrower-provided loan title. |
| desc | Borrower-provided free-text loan description. |

## 4) Loan Structure, Pricing, and Expected Risk
| Column | Description |
|---|---|
| loanAmnt | Requested/listed loan amount. |
| fundedAmnt | Amount committed/funded at that time. |
| term | Loan term in months (typically 36 or 60). |
| intRate | Stated interest rate. |
| effective_int_rate | Interest rate adjusted for expected uncollected interest before charge-off. |
| installment | Scheduled monthly payment if originated. |
| grade | Lending Club grade. |
| subGrade | Lending Club subgrade. |
| serviceFeeRate | Investor service fee rate. |
| expDefaultRate | Expected default rate. |
| disbursement_method | Disbursement method (CASH, DIRECT_PAY). |

## 5) Debt Burden and Credit Utilization
| Column | Description |
|---|---|
| dti | Debt-to-income ratio excluding mortgage and requested LC loan. |
| dti_joint | Joint debt-to-income ratio for co-borrowers (same exclusion logic). |
| all_util | Balance-to-limit ratio across all trades. |
| bcUtil | Bankcard utilization (balance to limit). |
| il_util | Installment utilization (balance to limit). |
| revolUtil | Revolving utilization rate. |
| percentBcGt75 | Percent of bankcard accounts above 75% utilization. |
| bcOpenToBuy | Total open-to-buy on revolving bankcards. |
| max_bal_bc | Max current balance on revolving accounts. |
| revolBal | Total revolving balance. |
| total_bal_il | Total current balance on installment accounts. |
| totalBalExMort | Total credit balance excluding mortgage. |
| totalBcLimit | Total bankcard credit limit/high credit. |
| total_rev_hi_lim | Total revolving high credit/limit. |
| total_il_high_credit_limit | Total installment high credit/limit. |
| tot_hi_cred_lim | Total high credit/limit across accounts. |
| avg_cur_bal | Average current balance across accounts. |
| tot_cur_bal | Total current balance across accounts. |
| revol_bal_joint | Joint revolving balance (net of duplicate balances). |

## 6) Credit File Age and Recency
| Column | Description |
|---|---|
| earliestCrLine | Date earliest reported credit line was opened. |
| mo_sin_old_rev_tl_op | Months since oldest revolving account opened. |
| mo_sin_rcnt_rev_tl_op | Months since most recent revolving account opened. |
| mo_sin_rcnt_tl | Months since most recent account opened. |
| mths_since_oldest_il_open | Months since oldest installment account opened. |
| mths_since_rcnt_il | Months since most recent installment account opened. |
| mthsSinceMostRecentInq | Months since most recent inquiry. |
| mthsSinceRecentBc | Months since most recent bankcard account opened. |
| mthsSinceLastDelinq | Months since last delinquency. |
| mthsSinceRecentLoanDelinq | Months since most recent personal finance delinquency. |
| mthsSinceRecentRevolDelinq | Months since most recent revolving delinquency. |
| mths_since_last_major_derog | Months since most recent 90+ day derogatory event. |
| mthsSinceLastRecord | Months since last public record. |

## 7) Account Structure and Trade Counts
| Column | Description |
|---|---|
| openAcc | Number of open credit lines. |
| totalAcc | Total credit lines in borrower file. |
| mortAcc | Number of mortgage accounts. |
| accOpenPast24Mths | Number of trades opened in past 24 months. |
| open_acc_6m | Number of open trades in past 6 months. |
| open_rv_12m | Revolving trades opened in past 12 months. |
| open_rv_24m | Revolving trades opened in past 24 months. |
| open_il_12m | Installment accounts opened in past 12 months. |
| open_il_24m | Installment accounts opened in past 24 months. |
| open_act_il | Currently active installment trades. |
| num_tl_op_past_12m | Number of accounts opened in past 12 months. |
| num_rev_accts | Number of revolving accounts. |
| num_op_rev_tl | Number of open revolving trades. |
| num_actv_rev_tl | Number of currently active revolving trades. |
| num_rev_tl_bal_gt_0 | Revolving trades with balance > 0. |
| num_bc_tl | Number of bankcard accounts. |
| num_actv_bc_tl | Number of currently active bankcard accounts. |
| num_bc_sats | Number of satisfactory bankcard accounts. |
| num_il_tl | Number of installment accounts. |
| num_sats | Number of satisfactory accounts. |
| total_cu_tl | Number of finance trades. |

## 8) Inquiries, Delinquency, and Adverse Events
| Column | Description |
|---|---|
| inqLast6Mths | Credit inquiries in past 6 months (excluding auto and mortgage). |
| inq_last_12m | Credit inquiries in past 12 months. |
| inq_fi | Personal finance inquiries. |
| delinq2Yrs | Number of 30+ day delinquencies in past 2 years. |
| accNowDelinq | Number of accounts currently delinquent. |
| delinqAmnt | Past-due amount on currently delinquent accounts. |
| num_tl_30dpd | Number of accounts currently 30 days past due (recently updated). |
| num_tl_120dpd_2m | Number of accounts currently 120 days past due (recently updated). |
| num_tl_90g_dpd_24m | Number of accounts 90+ days past due in last 24 months. |
| num_accts_ever_120_pd | Number of accounts ever 120+ days past due. |
| chargeoff_within_12_mths | Charge-offs in last 12 months. |
| collections_12_mths_ex_med | Collections in last 12 months excluding medical. |
| tot_coll_amt | Total collection amounts ever owed. |
| pct_tl_nvr_dlq | Percent of trades never delinquent. |
| pubRec | Number of derogatory public records. |
| pub_rec_bankruptcies | Number of public record bankruptcies. |
| tax_liens | Number of tax liens. |

## 9) Secondary Applicant (Joint Loans)
| Column | Description |
|---|---|
| sec_app_fico_range_low | Secondary applicant FICO lower bound at application. |
| sec_app_fico_range_high | Secondary applicant FICO upper bound at application. |
| sec_app_earliest_cr_line | Earliest credit line for secondary applicant. |
| sec_app_inq_last_6mths | Secondary applicant inquiries in last 6 months. |
| sec_app_mort_acc | Secondary applicant mortgage account count. |
| sec_app_open_acc | Secondary applicant open trade count. |
| sec_app_revol_util | Secondary applicant revolving utilization ratio. |
| sec_app_open_act_il | Secondary applicant active installment trades. |
| sec_app_num_rev_accts | Secondary applicant revolving account count. |
| sec_app_chargeoff_within_12_mths | Secondary applicant charge-offs in last 12 months. |
| sec_app_collections_12_mths_ex_med | Secondary applicant collections in last 12 months excluding medical. |
| sec_app_mths_since_last_major_derog | Months since secondary applicant's most recent major derogatory event. |

## Notes for Modeling
- Time variables should be parsed to datetime where applicable (`acceptD`, `listD`, `creditPullD`, etc.).
- High-cardinality text/categorical fields (`emp_title`, `title`, `desc`, `url`) need explicit encoding strategy.
- Joint applicant fields are structurally missing for individual applications; do not treat these nulls as random missingness.
- Several features are near-duplicates or highly correlated (credit limit and utilization families); check multicollinearity before linear modeling.


In [ ]:
# Set dataset path
data_path = 'loan.csv'

# Load dataset into a DataFrame
df = pd.read_csv(data_path)

C:\Users\jdblu\AppData\Local\Temp\ipykernel_28224\2872600915.py:7: DtypeWarning: Columns (19,47,55,112,123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Columns: 145 entries, id to settlement_term
dtypes: float64(105), int64(4), object(36)
memory usage: 2.4+ GB


In [7]:
# Preview dataframe
display(df.head())

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Get dataset shape and target column
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

Rows: 2,260,668 | Columns: 145


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Columns: 145 entries, id to settlement_term
dtypes: float64(105), int64(4), object(36)
memory usage: 2.4+ GB


## 4. Data Cleaning

Identify and handle data quality issues such as missing values, incorrect types, and inconsistent entries.

In [9]:
# Build feature summary table
feature_summary = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "n_unique": df.nunique(dropna=False).values,
})


# TODO: Check missing values by column
# HINT: df.isnull().sum()

# TODO: Decide strategy for each feature
# HINT: fillna() for imputation, dropna() for removals

# TODO: Handle data types (numeric, categorical, datetime)
# HINT: astype(), pd.to_numeric(), pd.to_datetime()

# TODO: Document your cleaning assumptions in markdown notes

In [11]:
# Grab columns where the percentage of missing values is greater than a certain threshold (e.g., 50%)
missing_threshold = 50.0
missing_columns = feature_summary.loc[feature_summary["missing_pct"] > missing_threshold, ["feature", "missing_pct"]].sort_values("missing_pct", ascending=False)
print(f"Features with missing_pct > {missing_threshold}%: {len(missing_columns)}")
display(missing_columns.reset_index(drop=True))


Features with missing_pct > 50.0%: 44


,feature,missing_pct
0,id,100.00
1,url,100.00
2,member_id,100.00
3,orig_projected_additional_accrued_interest,99.63
4,hardship_length,99.53
5,hardship_reason,99.53
6,hardship_status,99.53
7,deferral_term,99.53
8,hardship_amount,99.53
9,hardship_start_date,99.53


Columns with greater than 50% of their values missing (null) are generally considered poor candidates for analysis because they contain more missing information than observed data. With such a large proportion of absent values, any patterns or relationships derived from the feature are likely to be unreliable and heavily influenced by imputation assumptions rather than actual data. Additionally, imputing a majority of the values could introduce significant bias and noise, potentially distorting model performance.

In [14]:
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
non_numeric_summary = feature_summary.loc[
    feature_summary["feature"].isin(non_numeric_cols),
    ["feature", "dtype", "n_unique"]
]
print("\nTop 10 non-numeric columns by unique count:")
display(non_numeric_summary.sort_values("n_unique", ascending=False).head(10))


Top 10 non-numeric columns by unique count:


,feature,dtype,n_unique
10,emp_title,object,512695
19,desc,object,124501
21,title,object,63155
22,zip_code,object,957
26,earliest_cr_line,object,755
112,sec_app_earliest_cr_line,object,664
48,last_credit_pull_d,object,141
15,issue_d,object,139
45,last_pymnt_d,object,136
47,next_pymnt_d,object,106


In [18]:
# View low-cardinality categorical fields (good one-hot encoding candidates)
max_unique_for_ohe = 15
low_cardinality_summary = non_numeric_summary.loc[
    non_numeric_summary["n_unique"] <= max_unique_for_ohe
].sort_values(["n_unique"], ascending=[True])

print(f"\nCategorical fields with n_unique <= {max_unique_for_ohe}:")
display(low_cardinality_summary.reset_index(drop=True))


Categorical fields with n_unique <= 15:


,feature,dtype,n_unique
0,term,object,2
1,disbursement_method,object,2
2,hardship_type,object,2
3,hardship_flag,object,2
4,debt_settlement_flag,object,2
5,initial_list_status,object,2
6,application_type,object,2
7,pymnt_plan,object,2
8,verification_status,object,3
9,verification_status_joint,object,4


## 5. Exploratory Data Analysis (EDA)

Explore patterns in the data, with a special focus on the target distribution and feature behavior across default classes.

In [ ]:
# TODO: Analyze target distribution and class imbalance
# HINT: df['default'].value_counts(normalize=True)

# TODO: Univariate analysis for numeric features
# HINT: hist(), boxplot(), describe()

# TODO: Univariate analysis for categorical features
# HINT: value_counts(), seaborn.countplot

# TODO: Compare features by target class
# HINT: groupby('default'), seaborn.boxplot/histplot

## 6. Statistical Testing

Use statistical hypothesis tests to assess whether observed group differences are likely meaningful.

Interpretation reminder:
- Define a significance level (for example, alpha = 0.05).
- Compare p-value against alpha.
- Explain practical meaning, not only statistical significance.

### 6.1 T-Tests

Compare mean values of a numeric feature between default and non-default groups.

In [ ]:
# TODO: Select a numeric feature to test
feature_name = 'your_numeric_feature'

# TODO: Split feature values by target class
# HINT: df[df['default'] == 1][feature_name] and df[df['default'] == 0][feature_name]

# TODO: Run independent t-test
# HINT: from scipy.stats import ttest_ind
# HINT: t_stat, p_value = ttest_ind(group_default, group_non_default, equal_var=False)

# TODO: Interpret p-value in markdown
# TODO: State whether you reject or fail to reject the null hypothesis

### 6.2 ANOVA

Compare means across more than two groups for a numeric outcome.

In [ ]:
# TODO: Choose a grouping feature with 3+ categories and a numeric measurement
group_col = 'your_group_column'
value_col = 'your_numeric_feature'

# TODO: Build group arrays for ANOVA
# HINT: [df[df[group_col] == g][value_col].dropna() for g in df[group_col].unique()]

# TODO: Run ANOVA
# HINT: from scipy.stats import f_oneway
# HINT: f_stat, p_value = f_oneway(*group_arrays)

# TODO: Interpret p-value and discuss which post-hoc analysis you might run next

### 6.3 Chi-Square Test

Assess independence between two categorical variables.

In [ ]:
# TODO: Choose two categorical columns
cat_col_1 = 'your_categorical_feature_1'
cat_col_2 = 'default'

# TODO: Build a contingency table
# HINT: pd.crosstab(df[cat_col_1], df[cat_col_2])

# TODO: Run chi-square test
# HINT: from scipy.stats import chi2_contingency
# HINT: chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# TODO: Interpret p-value and explain whether variables appear independent

## 7. Feature Engineering

Create model-ready features through transformation, encoding, and scaling.

In [ ]:
# TODO: Create new features (example: financial ratios)
# HINT: debt_to_income = debt / income (handle division by zero carefully)

# TODO: Encode categorical variables
# HINT: pd.get_dummies() or sklearn.preprocessing.OneHotEncoder

# TODO: Scale numeric variables where appropriate
# HINT: sklearn.preprocessing.StandardScaler

# TODO: Keep a record of feature transformations for reproducibility

## 8. Train-Test Split

Split your engineered dataset into train and test sets before model training.

In [ ]:
# TODO: Separate features and target
# HINT: X = ..., y = df['default']

# TODO: Perform train-test split
# HINT: from sklearn.model_selection import train_test_split
# HINT: Use stratify=y if class imbalance exists

X_train, X_test, y_train, y_test = None, None, None, None

## 9. Baseline Model (Logistic Regression)

Train a baseline classifier and generate initial predictions.

In [ ]:
# TODO: Initialize logistic regression model
# HINT: from sklearn.linear_model import LogisticRegression

# TODO: Fit model on training data

# TODO: Generate class predictions and prediction probabilities
# HINT: model.predict(X_test), model.predict_proba(X_test)[:, 1]

baseline_model = None
y_pred_log = None
y_proba_log = None

## 10. Regularization

Compare L1 and L2 regularized logistic regression to evaluate sparsity and stability effects.

In [ ]:
# TODO: Train L1-regularized logistic regression
# HINT: penalty='l1', choose compatible solver (for example, 'liblinear' or 'saga')

# TODO: Train L2-regularized logistic regression
# HINT: penalty='l2'

# TODO: Compare coefficients between L1 and L2 models
# HINT: Inspect model.coef_ values and count near-zero coefficients

l1_model = None
l2_model = None

## 11. Tree-Based Models

Train nonlinear models and compare them with logistic regression baseline.

In [ ]:
# TODO: Train Random Forest classifier
# HINT: from sklearn.ensemble import RandomForestClassifier

# TODO (Optional): Train Gradient Boosting classifier
# HINT: from sklearn.ensemble import GradientBoostingClassifier

rf_model = None
gb_model = None  # Optional

## 12. Model Evaluation

Evaluate model quality with confusion matrix and classification metrics focused on default-risk use cases.

In [ ]:
# TODO: Compute confusion matrix for each model
# HINT: sklearn.metrics.confusion_matrix

# TODO: Calculate precision, recall, and F1
# HINT: sklearn.metrics.classification_report or precision_score/recall_score/f1_score

# TODO: Compare where false negatives vs false positives matter most for this problem

## 13. ROC Curve and AUC

### What ROC Represents
The ROC curve shows the trade-off between True Positive Rate and False Positive Rate across classification thresholds.

### Why AUC Is Useful
AUC summarizes ranking performance across all thresholds into one score. Higher AUC generally indicates better class separation.

In [ ]:
# TODO: Compute ROC curve coordinates
# HINT: from sklearn.metrics import roc_curve

# TODO: Compute AUC score
# HINT: from sklearn.metrics import roc_auc_score

# TODO: Plot ROC curves for multiple models on the same figure
# HINT: matplotlib.pyplot.plot

## 14. Feature and Hyperparameter Selection

Define a strategy to select the most useful features and tune key model hyperparameters in a reproducible way.

Suggested approach:
- Start with a small set of candidate models.
- Use cross-validation to compare settings.
- Track both model performance and interpretability.

### Tuning Run Log (Template)
Use this compact table to record each run.

| Run ID | Model | Feature Set | Search Method | Hyperparameters Tested | CV Metric | Mean CV Score | Validation/Test Score | Notes |
|---|---|---|---|---|---|---|---|---|
| 1 | TODO | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| 2 | TODO | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| 3 | TODO | TODO | TODO | TODO | TODO | TODO | TODO | TODO |

In [ ]:
# TODO: Define candidate feature subsets
# HINT: Try baseline feature set vs engineered feature set

# TODO: Define hyperparameter grids for one or more models
# HINT: Use sklearn.model_selection.GridSearchCV or RandomizedSearchCV
# HINT: Tune examples: C (logistic regression), max_depth/n_estimators (random forest)

# TODO: Choose a scoring metric aligned with business goals
# HINT: precision, recall, f1, or roc_auc

# TODO: Run cross-validated search on training data only
# TODO: Record best params and best cross-validation score for each model

best_params_summary = None


## 15. Model Comparison

Create a side-by-side summary of all candidate models and select a final approach.

In [ ]:
# TODO: Build a comparison table for key metrics
# HINT: Include Precision, Recall, F1, AUC, and notes on interpretability

# TODO: Select the best model based on project priorities
# TODO: Justify your selection in 2-4 concise bullet points

## 16. Feature Importance and Interpretation

Interpret which features most strongly influence predicted default risk.

In [ ]:
# TODO: Extract feature importance or coefficient-based influence
# HINT: RandomForestClassifier.feature_importances_
# HINT: LogisticRegression.coef_ (check sign and magnitude)

# TODO: Visualize top features
# HINT: bar chart with sorted importance values

# TODO: Interpret key drivers in plain business language

## 17. Final Insights and Business Recommendations

### TODO: Summarize Findings
- What patterns were most predictive of default?
- Which model performed best and why?

### TODO: Translate to Business Impact
- How could this model support lending decisions?
- What is the trade-off between catching defaults and rejecting safe applicants?
- What monitoring or retraining strategy would you propose for production use?